In [1]:
import numpy as np
import torch.nn as nn
import torch
import gymnasium as gym

from torch.distributions import Categorical
import torch.nn.functional as F

In [16]:
# VARIABLES
OBSERVATION_DIM = 22
ACTION_DIM = 19

NUM_ENVS = 32
TRAJECTORY_WINDOW = 128

CLIP = 0.2
VF_COEF = 0.5
ENT_COEF = 0.01
EPOCHS = 10
MB_SIZE = 256

GAMMA = 0.99
LAMBDA = 0.95

NUM_EPISODES = 25

In [17]:
class ActorCritic(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden_dim=64):
        super().__init__()
        # "new actor"
        self.actor = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, act_dim),
        )

        self.critic = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, obs):
        action_probs = self.actor(obs)
        value = self.critic(obs)
        return action_probs, value

In [10]:
import sys
from pathlib import Path

# Ensure parent project folder is importable in this notebook kernel
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
	sys.path.append(str(project_root))

from core.environment import Environment
from core.configs import EnvironmentConfig

torch.manual_seed(42)

cfg = EnvironmentConfig(num_agents=2, SEE_CARD_COUNTS=True)
env = Environment(cfg)

In [18]:
ppomodel = ActorCritic(obs_dim=OBSERVATION_DIM, act_dim=ACTION_DIM)
criterion = nn.MSELoss()

In [19]:
# Collect trajectories
observations = []
actions = []
rewards = []
values = []

obs, _ = env.reset()
state = obs["observation"]
action_mask = obs["action_mask"]
claim_sequence = obs["claim_seq"]
for episode in range(NUM_EPISODES):
    with torch.no_grad():
        logits = ppomodel.actor(state)
        action = F.softmax(logits)
        action = action * action_mask  # Apply action mask
        action = action / action.sum()  # Normalize

        next_obs, rewards, terminated, truncated, info = env.step(action)



C:\Users\derek\AppData\Local\Temp\ipykernel_204436\3239296413.py:14: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  action = F.softmax(logits)


ValueError: Expected SelectCardAction in SELECT phase, got <class 'core.models.StartClaimAction'>

In [23]:
obs, _ = env.reset()
state = obs["observation"]
action_mask = obs["action_mask"]
claim_sequence = obs["claim_seq"]

print("Initial observation:", state.shape)
print("State:", action_mask.shape)

with torch.no_grad():
    logits = ppomodel.actor(state)
    print("Logits:", logits)
    masked_logits = logits.masked_fill(action_mask == 0, float("-inf"))
    print("Masked logits:", masked_logits)
    probs = torch.softmax(masked_logits, dim=-1)
    print("Action probabilities:", probs)
    dist = torch.distributions.Categorical(probs)
    action = dist.sample()
    print("Sampled action:", action)

    next_obs, rewards, terminated, truncated, info = env.step(action)
    print("Next observation:", next_obs)
    print("Rewards:", rewards)
    print("Terminated:", terminated)
    print("Truncated:", truncated)

Initial observation: torch.Size([22])
State: torch.Size([19])
Logits: tensor([-0.4861,  0.6230,  0.0322, -0.0013,  0.3485, -0.4001, -0.0566,  0.0035,
        -0.3349, -1.0263, -0.2791,  0.1862, -0.6736,  0.9174, -0.1077, -0.3682,
         0.1555, -0.0453, -1.0474])
Masked logits: tensor([-0.4861,  0.6230,  0.0322, -0.0013,    -inf,    -inf,    -inf,    -inf,
           -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
           -inf,    -inf,    -inf])
Action probabilities: tensor([0.1363, 0.4133, 0.2289, 0.2214, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000])
Sampled action: tensor(1)
Next observation: {'observation': tensor([ 0.0000,  1.0000,  0.0000,  2.0000,  1.0000,  1.0000,  1.0000,  3.0000,
         0.0000,  3.0000,  1.0000,  2.0000,  3.0000,  4.0000,  2.0000,  3.0000,
         0.0000,  0.0769,  0.0000,  0.0000, 26.0000, 26.0000]), 'action_mask': tensor([0, 0, 0, 0, 1, 1,